# 02 - Construcción de staging limpio ENAHO 2025 Módulo 100

Este notebook construye una base staging limpia a partir del CSV raw del módulo 100 de ENAHO 2025 y del diccionario reducido del datamart.

La salida principal será:

`stg_enaho_hogar_100_limpio.csv`

Esta tabla todavía no corresponde al datamart final, sino a una base intermedia lista para alimentar dimensiones y tabla de hechos.


## Decisiones provenientes del data profiling

Este notebook aplica las decisiones obtenidas en el profiling del módulo 100:

1. Se carga el CSV con separador `;` y codificación `latin1`.
2. Todas las variables se cargan inicialmente como texto para evitar conversiones incorrectas.
3. Se usan solo las variables del diccionario reducido del datamart.
4. Las variables plantilla `I1172$XX`, `I1173$XX` e `I1174$XX` se expanden a las columnas reales existentes en el CSV.
5. Se filtran únicamente hogares con entrevista completa: `RESULT = 1`.
6. La llave única del hogar se construye con `AÑO`, `CONGLOME`, `VIVIENDA` y `HOGAR`.
7. Los códigos categóricos se traducen mediante reglas explícitas.
8. El código `P113A = 9` se conserva como categoría válida: `Bosta o estiércol`.
9. Los nulos derivados de saltos del cuestionario no se reemplazan automáticamente.
10. Se crean indicadores derivados alineados con la problemática de brecha multidimensional de bienestar habitacional.

In [80]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

PATH_PROFILED = Path.cwd().parent / "data" / "interim" / "enaho100_profiled.csv"

df_base = pd.read_csv(PATH_PROFILED)

print(df_base.shape)
df_base.head()

(27260, 159)


C:\Users\guill\AppData\Local\Temp\ipykernel_5796\1375005091.py:10: DtypeWarning: Columns (74,77,79,80,81,83,84,85,86,88,90) have mixed types. Specify dtype option on import or set low_memory=False.
  df_base = pd.read_csv(PATH_PROFILED)


,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,UBIGEO,DOMINIO,ESTRATO,PERIODO,FECENT,RESULT,P101,P102,P103,P103A,P104,P104A,P105A,P105B,P106,P106A,P106B,P107B1,P107D1,P107B2,P107D2,P107B3,P107D3,P107B4,P107D4,P110,P110A1,P110A,P110A_MODIFICADA,P110C,P110C1,P110C2,P110C3,P110D,P110E,P111A,P1121,P1123,P1124,P1125,P1126,P1127,P112A,P1131,P1132,P1133,P1135,P1136,P1139,P1137,P1138,P113A,P1141,P1142,P1143,...,I1174$05,I1173$06,I1174$06,I1173$07,I1174$07,I1173$08,I1174$08,I1173$09,I1174$09,I1173$10,I1174$10,I1173$11,I1174$11,I1173$12,I1174$12,I1173$13,I1174$13,I1173$14,I1174$14,I1173$15,I1174$15,I1173$16,I1174$16,I1173$17,I1174$17,T111A,NBI1,NBI2,NBI3,NBI4,NBI5,FACTOR07,CODCCPP,NOMCCPP,LONGITUD,LATITUD,ALTITUD,hogar_id,FACTOR07_num,P104_num,P104A_num,P105B_num,P106_num,I105B_num,I106_num,P110A_MODIFICADA_num,P110C1_num,P110C2_num,P110C3_num,P107D1_num,P107D2_num,P107D3_num,P107D4_num,D107D1_num,D107D2_num,D107D3_num,D107D4_num,LONGITUD_num,LATITUD_num,ALTITUD_num
0,2025,1,15009,12,11,10101,4,4,2,20250113,1,1.0,1.0,2.0,1.0,4.0,2.0,2,NaN,780.0,1.0,1.0,2,NaN,2,NaN,2,NaN,2,NaN,1,1.0,1,1,1.0,24.0,NaN,NaN,1,1,1,1,0,0,0,0,0,1.0,0,1,0,0,0,0,0,0,2.0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,"87,2856903076172",1,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301",2025_015009_012_11,87.28569,4.0,2.0,NaN,780.0,NaN,9347.0,1.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-77.869231,-6.234387,2350.557063
1,2025,1,15009,25,11,10101,4,4,2,20250114,1,1.0,3.0,5.0,4.0,3.0,3.0,2,NaN,225.0,2.0,NaN,2,NaN,2,NaN,2,NaN,2,NaN,1,1.0,1,",8",1.0,24.0,NaN,NaN,1,1,1,1,0,0,0,0,0,1.0,0,1,0,0,1,0,1,0,2.0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,336.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,1,0,0,0,0,0,"87,2856903076172",1,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301",2025_015009_025_11,87.28569,3.0,3.0,NaN,225.0,NaN,2696.0,0.8,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-77.869231,-6.234387,2350.557063
2,2025,1,15009,49,11,10101,4,4,2,20250125,1,1.0,1.0,5.0,1.0,3.0,2.0,2,NaN,785.0,1.0,1.0,2,NaN,2,NaN,2,NaN,2,NaN,1,1.0,1,",6",1.0,24.0,NaN,NaN,1,1,1,1,0,0,0,0,0,1.0,1,1,0,0,1,0,0,0,2.0,0,1,1,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,"87,2856903076172",1,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301",2025_015009_049_11,87.28569,3.0,2.0,NaN,785.0,NaN,9407.0,0.6,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-77.869231,-6.234387,2350.557063
3,2025,1,15009,61,11,10101,4,4,2,20250203,1,1.0,3.0,5.0,5.0,2.0,1.0,2,NaN,180.0,1.0,1.0,2,NaN,2,NaN,2,NaN,2,NaN,1,1.0,1,"1,2",1.0,24.0,NaN,NaN,1,1,1,1,0,0,0,0,0,2.0,0,1,0,0,0,0,0,0,2.0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,"87,2856903076172",1,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301",2025_015009_061_11,87.28569,2.0,1.0,NaN,180.0,NaN,2146.0,1.2,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-77.869231,-6.234387,2350.557063
4,2025,1,15009,74,11,10101,4,4,2,20250114,1,4.0,3.0,5.0,4.0,1.0,1.0,1,600.0,NaN,NaN,NaN,2,NaN,2,NaN,2,NaN,2,NaN,1,1.0,1,"1,2",1.0,24.0,NaN,NaN,1,1,2,1,0,0,0,0,0,2.0,1,1,0,0,0,0,0,0,2.0,1,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,0,0,0,0,"87,2856903076172",1,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301",2025_015009_074_11,87.28569,1.0,1.0,600.0,NaN,7190.0,NaN,1.2,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-77.869231,-6.234387,2350.557063


In [81]:
assert "hogar_id" in df_base.columns, "No existe hogar_id"
assert df_base["hogar_id"].duplicated().sum() == 0, "Hay hogares duplicados"

assert (
    df_base["RESULT"]
    .astype("string")
    .str.strip()
    .eq("1")
    .all()
), "Hay registros no completos"

print("Base intermedia cargada correctamente.")

Base intermedia cargada correctamente.


### 1. Funciones y diccionarios

In [82]:
def get_col(df, col_name):
    if col_name in df.columns:
        return df[col_name]
    return pd.Series(pd.NA, index=df.index, dtype="string")


def to_number(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace("", pd.NA)
        .pipe(pd.to_numeric, errors="coerce")
    )


def flag_in(series, positive_values, valid_values=None):
    s = series.astype("string").str.strip()
    positive_values = set(map(str, positive_values))

    if valid_values is None:
        valid_values = set(s.dropna().unique())
    else:
        valid_values = set(map(str, valid_values))

    result = pd.Series(pd.NA, index=s.index, dtype="Int64")
    result[s.isin(valid_values)] = 0
    result[s.isin(positive_values)] = 1

    return result


def map_category(series, mapping):
    s = series.astype("string").str.strip()
    return s.map(mapping).astype("string")


map_dominio = {
    "1": "Costa Norte", "1.0": "Costa Norte",
    "2": "Costa Centro", "2.0": "Costa Centro",
    "3": "Costa Sur", "3.0": "Costa Sur",
    "4": "Sierra Norte", "4.0": "Sierra Norte",
    "5": "Sierra Centro", "5.0": "Sierra Centro",
    "6": "Sierra Sur", "6.0": "Sierra Sur",
    "7": "Selva", "7.0": "Selva",
    "8": "Lima Metropolitana", "8.0": "Lima Metropolitana",
}

map_estrato = {
    "1": "De 500 000 a más habitantes", "1.0": "De 500 000 a más habitantes",
    "2": "De 100 000 a 499 999 habitantes", "2.0": "De 100 000 a 499 999 habitantes",
    "3": "De 50 000 a 99 999 habitantes", "3.0": "De 50 000 a 99 999 habitantes",
    "4": "De 20 000 a 49 999 habitantes", "4.0": "De 20 000 a 49 999 habitantes",
    "5": "De 2 000 a 19 999 habitantes", "5.0": "De 2 000 a 19 999 habitantes",
    "6": "De 500 a 1 999 habitantes", "6.0": "De 500 a 1 999 habitantes",
    "7": "Área de Empadronamiento Rural compuesto", "7.0": "Área de Empadronamiento Rural compuesto",
    "8": "Área de Empadronamiento Rural simple", "8.0": "Área de Empadronamiento Rural simple"
}

map_tipo_vivienda = {
    "1": "Casa independiente", "1.0": "Casa independiente",
    "2": "Departamento en edificio", "2.0": "Departamento en edificio",
    "3": "Vivienda en quinta", "3.0": "Vivienda en quinta",
    "4": "Vivienda en casa de vecindad", "4.0": "Vivienda en casa de vecindad",
    "5": "Choza o cabaña", "5.0": "Choza o cabaña",
    "6": "Vivienda improvisada", "6.0": "Vivienda improvisada",
    "7": "Local no destinado para habitación humana", "7.0": "Local no destinado para habitación humana",
    "8": "Otro", "8.0": "Otro"
}

map_pared = {
    "1": "Ladrillo o bloque de cemento", "1.0": "Ladrillo o bloque de cemento",
    "2": "Piedra o sillar con cal o cemento", "2.0": "Piedra o sillar con cal o cemento",
    "3": "Adobe", "3.0": "Adobe",
    "4": "Tapia", "4.0": "Tapia",
    "5": "Quincha", "5.0": "Quincha",
    "6": "Piedra con barro", "6.0": "Piedra con barro",
    "7": "Madera", "7.0": "Madera",
    "8": "Triplay, calamina o estera", "8.0": "Triplay, calamina o estera",
    "9": "Otro material", "9.0": "Otro material"
}

map_piso = {
    "1": "Parquet o madera pulida", "1.0": "Parquet o madera pulida",
    "2": "Láminas asfálticas, vinílicos o similares", "2.0": "Láminas asfálticas, vinílicos o similares",
    "3": "Losetas, terrazos o similares", "3.0": "Losetas, terrazos o similares",
    "4": "Madera", "4.0": "Madera",
    "5": "Cemento", "5.0": "Cemento",
    "6": "Tierra", "6.0": "Tierra",
    "7": "Otro material", "7.0": "Otro material"
}

map_techo = {
    "1": "Concreto armado", "1.0": "Concreto armado",
    "2": "Madera", "2.0": "Madera",
    "3": "Tejas", "3.0": "Tejas",
    "4": "Planchas de calamina, fibra de cemento o similares", "4.0": "Planchas de calamina, fibra de cemento o similares",
    "5": "Caña o estera con torta de barro o cemento", "5.0": "Caña o estera con torta de barro o cemento",
    "6": "Triplay, estera o carrizo", "6.0": "Triplay, estera o carrizo",
    "7": "Paja u hojas de palmera", "7.0": "Paja u hojas de palmera",
    "8": "Otro material", "8.0": "Otro material"
}

map_tenencia = {
    "1": "Alquilada", "1.0": "Alquilada",
    "2": "Propia totalmente pagada", "2.0": "Propia totalmente pagada",
    "3": "Propia por invasión", "3.0": "Propia por invasión",
    "4": "Propia comprándola a plazos", "4.0": "Propia comprándola a plazos",
    "5": "Cedida por centro de trabajo", "5.0": "Cedida por centro de trabajo",
    "6": "Cedida por otro hogar o institución", "6.0": "Cedida por otro hogar o institución",
    "7": "Otra forma", "7.0": "Otra forma"
}

map_fuente_agua = {
    "1": "Red pública dentro de la vivienda", "1.0": "Red pública dentro de la vivienda",
    "2": "Red pública fuera de la vivienda pero dentro del edificio", "2.0": "Red pública fuera de la vivienda pero dentro del edificio",
    "3": "Pilón o pileta de uso público", "3.0": "Pilón o pileta de uso público",
    "4": "Camión cisterna u otro similar", "4.0": "Camión cisterna u otro similar",
    "5": "Pozo", "5.0": "Pozo",
    "6": "Manantial o puquio", "6.0": "Manantial o puquio",
    "7": "Otra", "7.0": "Otra",
    "8": "Río, acequia, lago o laguna", "8.0": "Río, acequia, lago o laguna"
}

map_cloro = {
    "1": "Seguro", "1.0": "Seguro",
    "2": "Inadecuada dosificación de cloro", "2.0": "Inadecuada dosificación de cloro",
    "3": "Sin cloro", "3.0": "Sin cloro"
}

map_saneamiento = {
    "1": "Red pública de desagüe dentro de la vivienda", "1.0": "Red pública de desagüe dentro de la vivienda",
    "2": "Red pública de desagüe fuera de la vivienda", "2.0": "Red pública de desagüe fuera de la vivienda",
    "3": "Letrina con tratamiento", "3.0": "Letrina con tratamiento",
    "4": "Pozo séptico, tanque séptico o biodigestor", "4.0": "Pozo séptico, tanque séptico o biodigestor",
    "5": "Pozo ciego o negro", "5.0": "Pozo ciego o negro",
    "6": "Río, acequia, canal o similar", "6.0": "Río, acequia, canal o similar",
    "7": "Otra", "7.0": "Otra",
    "9": "Campo abierto o al aire libre", "9.0": "Campo abierto o al aire libre",
    "10": "Letrina sin tratamiento", "10.0": "Letrina sin tratamiento",
    "11": "Letrina tipo compostera", "11.0": "Letrina tipo compostera"
}

map_combustible = {
    "1": "Electricidad", "1.0": "Electricidad",
    "2": "Gas balón GLP", "2.0": "Gas balón GLP",
    "3": "Gas natural", "3.0": "Gas natural",
    "5": "Carbón", "5.0": "Carbón",
    "6": "Leña", "6.0": "Leña",
    "7": "Otro", "7.0": "Otro",
    "8": "No cocinan", "8.0": "No cocinan",
    "9": "Bosta o estiércol", "9.0": "Bosta o estiércol"
}

### 2. Construcción de staging limpio

In [83]:
stg = pd.DataFrame(index=df_base.index)

# ============================================================
# Identificación
# ============================================================

stg["hogar_id"] = get_col(df_base, "hogar_id")

stg["anio"] = get_col(df_base, "AÑO")
stg["mes"] = get_col(df_base, "MES")
stg["periodo"] = get_col(df_base, "PERIODO")

stg["conglome"] = get_col(df_base, "CONGLOME")
stg["vivienda"] = get_col(df_base, "VIVIENDA")
stg["hogar"] = get_col(df_base, "HOGAR")

# ============================================================
# Ubicación y muestra
# ============================================================

stg["ubigeo"] = get_col(df_base, "UBIGEO")

stg["dominio_cod"] = get_col(df_base, "DOMINIO")
stg["dominio"] = map_category(get_col(df_base, "DOMINIO"), map_dominio)

stg["estrato_cod"] = get_col(df_base, "ESTRATO")
stg["estrato"] = map_category(get_col(df_base, "ESTRATO"), map_estrato)

stg["factor_expansion"] = to_number(get_col(df_base, "FACTOR07"))

stg["latitud"] = to_number(get_col(df_base, "LATITUD"))
stg["longitud"] = to_number(get_col(df_base, "LONGITUD"))
stg["altitud"] = to_number(get_col(df_base, "ALTITUD"))

# ============================================================
# Vivienda
# ============================================================

stg["tipo_vivienda_cod"] = get_col(df_base, "P101")
stg["tipo_vivienda"] = map_category(get_col(df_base, "P101"), map_tipo_vivienda)

stg["material_pared_cod"] = get_col(df_base, "P102")
stg["material_pared"] = map_category(get_col(df_base, "P102"), map_pared)

stg["material_piso_cod"] = get_col(df_base, "P103")
stg["material_piso"] = map_category(get_col(df_base, "P103"), map_piso)

stg["material_techo_cod"] = get_col(df_base, "P103A")
stg["material_techo"] = map_category(get_col(df_base, "P103A"), map_techo)

stg["tenencia_vivienda_cod"] = get_col(df_base, "P105A")
stg["tenencia_vivienda"] = map_category(get_col(df_base, "P105A"), map_tenencia)

stg["habitaciones_total"] = to_number(get_col(df_base, "P104"))
stg["habitaciones_dormir"] = to_number(get_col(df_base, "P104A"))

stg["alquiler_mensual_reportado"] = to_number(get_col(df_base, "P105B"))
stg["alquiler_estimado_mensual"] = to_number(get_col(df_base, "P106"))

stg["alquiler_anual_imputado"] = to_number(get_col(df_base, "I105B"))
stg["alquiler_estimado_anual_imputado"] = to_number(get_col(df_base, "I106"))

# ============================================================
# Agua, saneamiento, energía y conectividad
# ============================================================

stg["fuente_agua_cod"] = get_col(df_base, "P110")
stg["fuente_agua"] = map_category(get_col(df_base, "P110"), map_fuente_agua)

stg["nivel_cloro_cod"] = get_col(df_base, "P110A")
stg["nivel_cloro"] = map_category(get_col(df_base, "P110A"), map_cloro)

stg["servicio_higienico_cod"] = get_col(df_base, "T111A")
stg["servicio_higienico"] = map_category(get_col(df_base, "T111A"), map_saneamiento)

stg["combustible_cocina_cod"] = get_col(df_base, "P113A")
stg["combustible_cocina"] = map_category(get_col(df_base, "P113A"), map_combustible)

stg.head()

,hogar_id,anio,mes,periodo,conglome,vivienda,hogar,ubigeo,dominio_cod,dominio,estrato_cod,estrato,factor_expansion,latitud,longitud,altitud,tipo_vivienda_cod,tipo_vivienda,material_pared_cod,material_pared,material_piso_cod,material_piso,material_techo_cod,material_techo,tenencia_vivienda_cod,tenencia_vivienda,habitaciones_total,habitaciones_dormir,alquiler_mensual_reportado,alquiler_estimado_mensual,alquiler_anual_imputado,alquiler_estimado_anual_imputado,fuente_agua_cod,fuente_agua,nivel_cloro_cod,nivel_cloro,servicio_higienico_cod,servicio_higienico,combustible_cocina_cod,combustible_cocina
0,2025_015009_012_11,2025,1,2,15009,12,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,1.0,Ladrillo o bloque de cemento,2.0,"Láminas asfálticas, vinílicos o similares",1.0,Concreto armado,2,Propia totalmente pagada,4.0,2.0,<NA>,780.0,<NA>,9347.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP
1,2025_015009_025_11,2025,1,2,15009,25,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,3.0,Adobe,5.0,Cemento,4.0,"Planchas de calamina, fibra de cemento o simil...",2,Propia totalmente pagada,3.0,3.0,<NA>,225.0,<NA>,2696.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP
2,2025_015009_049_11,2025,1,2,15009,49,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,1.0,Ladrillo o bloque de cemento,5.0,Cemento,1.0,Concreto armado,2,Propia totalmente pagada,3.0,2.0,<NA>,785.0,<NA>,9407.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP
3,2025_015009_061_11,2025,1,2,15009,61,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,3.0,Adobe,5.0,Cemento,5.0,Caña o estera con torta de barro o cemento,2,Propia totalmente pagada,2.0,1.0,<NA>,180.0,<NA>,2146.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP
4,2025_015009_074_11,2025,1,2,15009,74,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,4.0,Vivienda en casa de vecindad,3.0,Adobe,5.0,Cemento,4.0,"Planchas de calamina, fibra de cemento o simil...",1,Alquilada,1.0,1.0,600.0,<NA>,7190.0,<NA>,1,Red pública dentro de la vivienda,1,Seguro,2,Red pública de desagüe fuera de la vivienda,2.0,Gas balón GLP


### 3. Indicadores derivados 

In [84]:
def normalize_code(series):
    """
    Convierte códigos tipo 1, 1.0, '1.0' a texto limpio '1'.
    Mantiene NA como NA.
    """
    return (
        pd.to_numeric(
            series.astype("string").str.strip().str.replace(",", ".", regex=False),
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

In [90]:
# ============================================================
# Materiales y habitabilidad
# ============================================================

stg["pared_precaria_ind"] = flag_in(
    normalize_code(get_col(df_base, "P102")),
    positive_values=["5", "6", "7", "8"],
    valid_values=["1", "2", "3", "4", "5", "6", "7", "8", "9"]
)

stg["piso_precario_ind"] = flag_in(
    normalize_code(get_col(df_base, "P103")),
    positive_values=["6"],
    valid_values=["1", "2", "3", "4", "5", "6", "7"]
)

stg["techo_precario_ind"] = flag_in(
    normalize_code(get_col(df_base, "P103A")),
    positive_values=["6", "7"],
    valid_values=["1", "2", "3", "4", "5", "6", "7", "8"]
)

stg["vivienda_material_precario_ind"] = (
    stg[["pared_precaria_ind", "piso_precario_ind", "techo_precario_ind"]]
    .fillna(0)
    .max(axis=1)
    .astype("Int64")
)

stg["nbi_vivienda_inadecuada_ind"] = to_number(get_col(df_base, "NBI1")).astype("Int64")
stg["nbi_hacinamiento_ind"] = to_number(get_col(df_base, "NBI2")).astype("Int64")
stg["nbi_sin_servicio_higienico_ind"] = to_number(get_col(df_base, "NBI3")).astype("Int64")
stg["nbi_ninios_no_asisten_ind"] = to_number(get_col(df_base, "NBI4")).astype("Int64")
stg["nbi_alta_dependencia_ind"] = to_number(get_col(df_base, "NBI5")).astype("Int64")

# ============================================================
# Agua y saneamiento
# ============================================================

stg["agua_red_publica_ind"] = flag_in(
    get_col(df_base, "P110"),
    positive_values=["1", "2"],
    valid_values=["1", "2", "3", "4", "5", "6", "7", "8"]
)

stg["agua_potable_ind"] = flag_in(
    normalize_code(get_col(df_base, "P110A1")),
    positive_values=["1"],
    valid_values=["1", "2"]
)

stg["cloro_seguro_ind"] = flag_in(
    get_col(df_base, "P110A"),
    positive_values=["1"],
    valid_values=["1", "2", "3"]
)

stg["agua_todos_dias_ind"] = flag_in(
    normalize_code(get_col(df_base, "P110C")),
    positive_values=["1"],
    valid_values=["1", "2", "9"]
)

# ============================================================
# Frecuencia y horas de abastecimiento de agua
# ============================================================

p110c = normalize_code(get_col(df_base, "P110C"))

print("Valores normalizados de P110C:")
print(p110c.value_counts(dropna=False).sort_index())

mask_agua_todos_dias = p110c.eq("1").fillna(False)
mask_agua_no_todos_dias = p110c.eq("2").fillna(False)

horas_todos_dias = to_number(get_col(df_base, "P110C1"))
dias_no_todos = to_number(get_col(df_base, "P110C2"))
horas_no_todos = to_number(get_col(df_base, "P110C3"))

stg["horas_agua_dia"] = pd.Series(pd.NA, index=df_base.index, dtype="Float64")
stg["dias_agua_semana"] = pd.Series(pd.NA, index=df_base.index, dtype="Float64")

# Si tiene agua todos los días
stg.loc[mask_agua_todos_dias, "horas_agua_dia"] = horas_todos_dias.loc[mask_agua_todos_dias]
stg.loc[mask_agua_todos_dias, "dias_agua_semana"] = 7

# Si no tiene agua todos los días
stg.loc[mask_agua_no_todos_dias, "dias_agua_semana"] = dias_no_todos.loc[mask_agua_no_todos_dias]
stg.loc[mask_agua_no_todos_dias, "horas_agua_dia"] = horas_no_todos.loc[mask_agua_no_todos_dias]

stg["agua_segura_ind"] = (
    (stg["agua_potable_ind"].fillna(0) == 1) &
    (stg["cloro_seguro_ind"].fillna(0) == 1)
).astype("Int64")

stg["saneamiento_adecuado_ind"] = flag_in(
    get_col(df_base, "T111A"),
    positive_values=["1", "2", "3", "4"],
    valid_values=["1","2","3","4","5","6","7","9","10","11"]
)

# ============================================================
# Energía
# ============================================================

stg["tiene_electricidad_ind"] = flag_in(
    get_col(df_base, "P1121"),
    positive_values=["1"],
    valid_values=["0", "1"]
)

stg["combustible_limpio_ind"] = flag_in(
    normalize_code(get_col(df_base, "P113A")),
    positive_values=["1", "2", "3"],
    valid_values=["1", "2", "3", "5", "6", "7", "8", "9"]
)


stg["combustible_solido_ind"] = flag_in(
    normalize_code(get_col(df_base, "P113A")),
    positive_values=["5", "6", "9"],
    valid_values=["1", "2", "3", "5", "6", "7", "8", "9"]
)

# ============================================================
# Conectividad
# ============================================================

stg["tiene_telefono_fijo_ind"] = flag_in(get_col(df_base, "P1141"), ["1"], ["0", "1"])
stg["tiene_celular_ind"] = flag_in(get_col(df_base, "P1142"), ["1"], ["0", "1"])
stg["tiene_tv_cable_ind"] = flag_in(get_col(df_base, "P1143"), ["1"], ["0", "1"])
stg["tiene_internet_ind"] = flag_in(get_col(df_base, "P1144"), ["1"], ["0", "1"])
stg["sin_tic_ind"] = flag_in(get_col(df_base, "P1145"), ["1"], ["0", "1"])
stg["tiene_tdt_ind"] = flag_in(get_col(df_base, "P1146"), ["1"], ["0", "1"])

stg.head()

Valores normalizados de P110C:
P110C
1       20197
2        3176
9           2
<NA>     3885
Name: count, dtype: Int64


,hogar_id,anio,mes,periodo,conglome,vivienda,hogar,ubigeo,dominio_cod,dominio,estrato_cod,estrato,factor_expansion,latitud,longitud,altitud,tipo_vivienda_cod,tipo_vivienda,material_pared_cod,material_pared,material_piso_cod,material_piso,material_techo_cod,material_techo,tenencia_vivienda_cod,tenencia_vivienda,habitaciones_total,habitaciones_dormir,alquiler_mensual_reportado,alquiler_estimado_mensual,alquiler_anual_imputado,alquiler_estimado_anual_imputado,fuente_agua_cod,fuente_agua,nivel_cloro_cod,nivel_cloro,servicio_higienico_cod,servicio_higienico,combustible_cocina_cod,combustible_cocina,pared_precaria_ind,piso_precario_ind,techo_precario_ind,vivienda_material_precario_ind,nbi_vivienda_inadecuada_ind,nbi_hacinamiento_ind,nbi_sin_servicio_higienico_ind,nbi_ninios_no_asisten_ind,nbi_alta_dependencia_ind,agua_red_publica_ind,agua_potable_ind,cloro_seguro_ind,agua_todos_dias_ind,horas_agua_dia,dias_agua_semana,tiene_electricidad_ind,combustible_limpio_ind,combustible_solido_ind,tiene_telefono_fijo_ind,tiene_celular_ind,tiene_tv_cable_ind,tiene_internet_ind,sin_tic_ind,tiene_tdt_ind,gasto_anual_servicios_pagado_hogar,gasto_anual_servicios_donado,gasto_anual_servicios_autoconsumo,gasto_anual_servicios_total,gasto_mensual_servicios_estimado,carencia_materiales_vivienda_ind,carencia_hacinamiento_ind,agua_segura_ind,saneamiento_adecuado_ind
0,2025_015009_012_11,2025,1,2,15009,12,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,1.0,Ladrillo o bloque de cemento,2.0,"Láminas asfálticas, vinílicos o similares",1.0,Concreto armado,2,Propia totalmente pagada,4.0,2.0,<NA>,780.0,<NA>,9347.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP,0,0,0,0,0,0,0,0,0,1,1,1,1,24.0,7.0,1,1,0,0,1,0,1,0,0,1763.0,0.0,0.0,1763.0,146.916667,0,0,1,1
1,2025_015009_025_11,2025,1,2,15009,25,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,3.0,Adobe,5.0,Cemento,4.0,"Planchas de calamina, fibra de cemento o simil...",2,Propia totalmente pagada,3.0,3.0,<NA>,225.0,<NA>,2696.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP,0,0,0,0,0,0,0,0,0,1,1,1,1,24.0,7.0,1,1,0,0,1,0,1,0,0,2268.0,336.0,0.0,2604.0,217.0,0,0,1,1
2,2025_015009_049_11,2025,1,2,15009,49,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,1.0,Ladrillo o bloque de cemento,5.0,Cemento,1.0,Concreto armado,2,Propia totalmente pagada,3.0,2.0,<NA>,785.0,<NA>,9407.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP,0,0,0,0,0,0,0,0,0,1,1,1,1,24.0,7.0,1,1,0,0,1,1,1,0,0,3363.0,0.0,0.0,3363.0,280.25,0,0,1,1
3,2025_015009_061_11,2025,1,2,15009,61,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,1.0,Casa independiente,3.0,Adobe,5.0,Cemento,5.0,Caña o estera con torta de barro o cemento,2,Propia totalmente pagada,2.0,1.0,<NA>,180.0,<NA>,2146.0,1,Red pública dentro de la vivienda,1,Seguro,1,Red pública de desagüe dentro de la vivienda,2.0,Gas balón GLP,0,0,0,0,0,0,0,0,0,1,1,1,1,24.0,7.0,1,1,0,0,1,0,1,0,0,346.0,476.0,0.0,822.0,68.5,0,0,1,1
4,2025_015009_074_11,2025,1,2,15009,74,11,10101,4,Sierra Norte,4,De 20 000 a 49 999 habitantes,87.28569,-6.234387,-77.869231,2350.557063,4.0,Vivienda en casa de vecindad,3.0,Adobe,5.0,Cemento,4.0,"Planchas de calamina, fibra de cemento o simil...",1,Alquilada,1.0,1.0,600.0,<NA>,7190.0,<NA>,1,Red pública dentro de la vivienda,1,Seguro,2,Red pública de desagüe fuera de la vivienda,2.0,Gas balón GLP,0,0,0,0,0,0,0,0,0,1,1,1,1,24.0,7.0,1,1,0,1,1,0,1,0,0,1881.0,0.0,0.0,1881.0,156.75,0,0,1,1


### 4. Gastos derivados de servicios de hogar

In [91]:
# ============================================================
# Gastos de servicios básicos
# ============================================================

cols_i1172 = [c for c in df_base.columns if c.startswith("I1172$")]
cols_i1173 = [c for c in df_base.columns if c.startswith("I1173$")]
cols_i1174 = [c for c in df_base.columns if c.startswith("I1174$")]

print("Columnas I1172 encontradas:", len(cols_i1172))
print("Columnas I1173 encontradas:", len(cols_i1173))
print("Columnas I1174 encontradas:", len(cols_i1174))

def sum_monetary_columns(df, columns):
    if len(columns) == 0:
        return pd.Series(0, index=df.index, dtype="float64")
    
    temp = pd.DataFrame(index=df.index)
    
    for col in columns:
        temp[col] = to_number(df[col])
    
    return temp.sum(axis=1, skipna=True)


stg["gasto_anual_servicios_pagado_hogar"] = sum_monetary_columns(df_base, cols_i1172)
stg["gasto_anual_servicios_donado"] = sum_monetary_columns(df_base, cols_i1173)
stg["gasto_anual_servicios_autoconsumo"] = sum_monetary_columns(df_base, cols_i1174)

stg["gasto_anual_servicios_total"] = (
    stg["gasto_anual_servicios_pagado_hogar"] +
    stg["gasto_anual_servicios_donado"] +
    stg["gasto_anual_servicios_autoconsumo"]
)

stg["gasto_mensual_servicios_estimado"] = (
    stg["gasto_anual_servicios_total"] / 12
)

stg[
    [
        "gasto_anual_servicios_pagado_hogar",
        "gasto_anual_servicios_donado",
        "gasto_anual_servicios_autoconsumo",
        "gasto_anual_servicios_total",
        "gasto_mensual_servicios_estimado"
    ]
].describe()

Columnas I1172 encontradas: 16
Columnas I1173 encontradas: 16
Columnas I1174 encontradas: 16


,gasto_anual_servicios_pagado_hogar,gasto_anual_servicios_donado,gasto_anual_servicios_autoconsumo,gasto_anual_servicios_total,gasto_mensual_servicios_estimado
count,27260.0,27260.0,27260.0,27260.0,27260.0
mean,1932.84973,96.698092,46.722744,2076.270567,173.022547
std,1603.036342,328.921895,170.96283,1579.24698,131.603915
min,0.0,0.0,0.0,0.0,0.0
25%,806.0,0.0,0.0,969.0,80.75
50%,1510.0,0.0,0.0,1661.0,138.416667
75%,2649.0,0.0,0.0,2769.0,230.75
max,14579.0,6452.0,3002.0,14579.0,1214.916667


### 5. Brecha multidimensional

In [92]:
# ============================================================
# Carencias alineadas con la problemática del proyecto
# ============================================================

stg["carencia_materiales_vivienda_ind"] = stg["vivienda_material_precario_ind"]
stg["carencia_hacinamiento_ind"] = stg["nbi_hacinamiento_ind"]

stg["carencia_agua_segura_ind"] = (stg["agua_segura_ind"] == 0).astype("Int64")
stg["carencia_saneamiento_ind"] = (stg["saneamiento_adecuado_ind"] == 0).astype("Int64")
stg["carencia_electricidad_ind"] = (stg["tiene_electricidad_ind"] == 0).astype("Int64")
stg["carencia_combustible_limpio_ind"] = (stg["combustible_limpio_ind"] == 0).astype("Int64")
stg["carencia_internet_ind"] = (stg["tiene_internet_ind"] == 0).astype("Int64")

cols_carencias = [
    "carencia_materiales_vivienda_ind",
    "carencia_hacinamiento_ind",
    "carencia_agua_segura_ind",
    "carencia_saneamiento_ind",
    "carencia_electricidad_ind",
    "carencia_combustible_limpio_ind",
    "carencia_internet_ind"
]

stg["total_carencias_bienestar_habitacional"] = (
    stg[cols_carencias]
    .fillna(0)
    .sum(axis=1)
    .astype("Int64")
)

stg["brecha_multidimensional_ind"] = (
    stg["total_carencias_bienestar_habitacional"] >= 2
).astype("Int64")

stg[
    [
        "total_carencias_bienestar_habitacional",
        "brecha_multidimensional_ind"
    ]
].value_counts().sort_index()

total_carencias_bienestar_habitacional  brecha_multidimensional_ind
0                                       0                              3883
1                                       0                              8321
2                                       1                              5368
3                                       1                              4880
4                                       1                              3028
5                                       1                              1254
6                                       1                               465
7                                       1                                61
Name: count, dtype: int64

### 6. Validaciones finales

In [93]:
print("Filas staging:", len(stg))
print("Columnas staging:", len(stg.columns))
print("Hogares únicos:", stg["hogar_id"].nunique())
print("Duplicados hogar_id:", stg["hogar_id"].duplicated().sum())

assert stg["hogar_id"].notna().all(), "Hay hogar_id nulos"
assert stg["hogar_id"].duplicated().sum() == 0, "Hay hogares duplicados"
assert stg["hogar_id"].nunique() == len(stg), "hogar_id no es único"

print("Validaciones principales correctas.")

Filas staging: 27260
Columnas staging: 80
Hogares únicos: 27260
Duplicados hogar_id: 0
Validaciones principales correctas.


### 7. Revisar indicadores

In [94]:
indicadores_revision = [
    "vivienda_material_precario_ind",
    "nbi_hacinamiento_ind",
    "agua_red_publica_ind",
    "agua_segura_ind",
    "saneamiento_adecuado_ind",
    "tiene_electricidad_ind",
    "combustible_limpio_ind",
    "tiene_internet_ind",
    "brecha_multidimensional_ind"
]

for col in indicadores_revision:
    print("\n" + "=" * 60)
    print(col)
    print(stg[col].value_counts(dropna=False).sort_index())


vivienda_material_precario_ind
vivienda_material_precario_ind
0    16302
1    10958
Name: count, dtype: Int64

nbi_hacinamiento_ind
nbi_hacinamiento_ind
0    26073
1     1187
Name: count, dtype: Int64

agua_red_publica_ind
agua_red_publica_ind
0     4383
1    22877
Name: count, dtype: Int64

agua_segura_ind
agua_segura_ind
0    21841
1     5419
Name: count, dtype: Int64

saneamiento_adecuado_ind
saneamiento_adecuado_ind
0     7573
1    19687
Name: count, dtype: Int64

tiene_electricidad_ind
tiene_electricidad_ind
0     1900
1    25360
Name: count, dtype: Int64

combustible_limpio_ind
combustible_limpio_ind
0        9042
1       17627
<NA>      591
Name: count, dtype: Int64

tiene_internet_ind
tiene_internet_ind
0     2795
1    24465
Name: count, dtype: Int64

brecha_multidimensional_ind
brecha_multidimensional_ind
0    12204
1    15056
Name: count, dtype: Int64


### 8. Guardar staging

In [95]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_STG_CSV = OUTPUT_DIR / "stg_enaho_hogar_100_limpio.csv"
PATH_STG_PARQUET = OUTPUT_DIR / "stg_enaho_hogar_100_limpio.parquet"

stg.to_csv(
    PATH_STG_CSV,
    index=False,
    encoding="utf-8-sig"
)

stg.to_parquet(
    PATH_STG_PARQUET,
    index=False
)

print("Staging CSV guardado en:")
print(PATH_STG_CSV)

print("\nStaging Parquet guardado en:")
print(PATH_STG_PARQUET)

Staging CSV guardado en:
..\data\processed\stg_enaho_hogar_100_limpio.csv

Staging Parquet guardado en:
..\data\processed\stg_enaho_hogar_100_limpio.parquet
